# 07. Auditoría experimental

La numeración organiza la exposición metodológica, no inventa una cronología nueva de experimentos.

**Dataset principal:** `data/nuevo_dataset/cardio_train.csv`. Este cuaderno no ejecuta entrenamientos. Los resultados históricos se leen de archivos conservados; no se sustituyen por nuevas ejecuciones.

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/nuevo_dataset/cardio_train.csv").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
from src.experimento import (
    ORIGINAL, RESULTADOS, VARIABLES, NOMBRES, cargar_datos, reglas_calidad,
    cargar_metricas, comparar_ejecuciones, tabla_configuracion, verificar_configuracion,
    construir_cnn, tabla_modelo, dibujar_arquitectura, guardar_figura_nueva,
    importancia_guardada, resumen_dataset, verificar_predicciones, tensorflow_disponible,
)

In [2]:
display(comparar_ejecuciones())

,Protocolo,Entrenamiento,Validación,Prueba,Épocas,Parámetros,Accuracy,Precision,Recall,F1,ROC-AUC
0,StandardScaler,10500,2250,2250,44,34529,0.712444,0.716141,0.693621,0.704701,0.775738
1,MinMaxScaler,10200,2550,2250,66,34529,0.731111,0.732601,0.718778,0.725624,0.788458


## Misma arquitectura, protocolos diferentes
Los antiguos notebooks 08 y 09 usaron exactamente la misma arquitectura de 34,529 parámetros. StandardScaler frente a MinMaxScaler, la selección de particiones, la reserva de validación, las métricas de compilación y la paciencia de parada cambiaron. Los 44 y 66 son épocas ejecutadas, no necesariamente las de los pesos restaurados. La comparación no aísla una modificación arquitectónica ni el efecto causal del escalador.

## Fuga de información y reproducibilidad
En la ejecución inicial StandardScaler se ajustó solo sobre entrenamiento. En la definitiva MinMaxScaler se ajustó sobre el bloque de 12,750 registros antes de reservar la validación interna. Esa validación no fue estratificada explícitamente. La prueba de 2,250 quedó fuera del ajuste del escalador. El notebook 04 describe un procedimiento futuro separado y sin esa influencia de validación, sin adjudicarle las métricas existentes.

No se guardaron el escalador definitivo ni identificadores explícitos de las particiones. La muestra y el código original están conservados; no se afirma una reproducibilidad bit a bit de un nuevo entrenamiento. Las coincidencias sin `id` requieren revisión específica y no se resuelven eliminando registros retrospectivamente.

## Alcance de las comparaciones
Las pruebas de ambas ejecuciones se generaron con procedimientos distintos; no basta con que tengan igual tamaño. Solo hay una ejecución por protocolo y no se estimó variabilidad entre semillas ni intervalos de confianza. Los resultados del dataset histórico no son una referencia numérica comparable con el dataset definitivo. La importancia del árbol usa otra partición 80/20 y no es una atribución de la CNN.

In [3]:
display(verificar_configuracion())

,capa,tipo,salida,parametros,filtros,kernel,activacion,padding,unidades,dropout
0,conv1d,Conv1D,"(11, 32)",128,32.0,3.0,relu,same,NaN,NaN
1,conv1d_1,Conv1D,"(11, 96)",9312,96.0,3.0,relu,same,NaN,NaN
2,global_average_pooling1d,GlobalAveragePooling1D,"(96,)",0,NaN,NaN,—,—,NaN,NaN
3,dense,Dense,"(256,)",24832,NaN,NaN,relu,—,256.0,NaN
4,dropout,Dropout,"(256,)",0,NaN,NaN,—,—,NaN,0.3
5,dense_1,Dense,"(1,)",257,NaN,NaN,sigmoid,—,1.0,NaN
